In [ ]:
from utils.logger import logger
from utils.config import config
from pipeline.pdf_parser import GrobidPDFParser
from pipeline.sentence_extractor import extract_sentences
from pipeline.reference_resolver import ReferenceResolver
from pipeline.classifier import CitationClassifier
from pipeline.retriever import HybridRetriever
from pipeline.urgency_scorer import UrgencyScorer
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from database.qdrant.store import create_qdrant_client

In [2]:
logger.info("Starting the application...")

logger.info("Parsing the PDF and extracting information...")

parser = GrobidPDFParser(pdf_path="../papers/BERT.pdf")
parsed_paper = parser.parse()

logger.info("Successfully parsed the PDF. Extracted information:")
logger.info(f"Title: {parsed_paper.title}")
logger.info(f"Abstract: {parsed_paper.abstract}")

2026-05-10 12:32:46,644 - missing_citations - INFO - Starting the application...
2026-05-10 12:32:46,646 - missing_citations - INFO - Parsing the PDF and extracting information...
2026-05-10 12:32:53,036 - missing_citations - INFO - Successfully parsed the PDF. Extracted information:
2026-05-10 12:32:53,039 - missing_citations - INFO - Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
2026-05-10 12:32:53,041 - missing_citations - INFO - Abstract: We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a;[CITE:b36], BERT is designed to pretrain deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be finetuned with just one additional output layer to create state-of-the-art models for a wide ra

In [3]:
logger.info("Extracting sentences from the parsed paper...")
sentences = extract_sentences(parsed_paper)
logger.info(f"Extracted {len(sentences)} sentences from the paper.")


2026-05-10 12:32:53,056 - missing_citations - INFO - Extracting sentences from the parsed paper...
2026-05-10 12:32:56,147 - missing_citations - INFO - Extracted 287 sentences from the paper.


In [4]:
logger.info("Resolving references in the paper...")
resolver = ReferenceResolver()
resolved_references = []

for ref in parsed_paper.references:
    resolved = resolver.resolve(ref)
    resolved_references.append(resolved)

logger.info("Resolved references:")
for ref, resolved in zip(parsed_paper.references, resolved_references):
    logger.info(f"Original: {ref}")
    logger.info(f"Resolved: {resolved}")
    print("---")

logger.info(f"Stats: {resolver.stats}")


2026-05-10 12:32:56,159 - missing_citations - INFO - Resolving references in the paper...
2026-05-10 12:33:43,564 - missing_citations - INFO - Resolved references:
2026-05-10 12:33:43,565 - missing_citations - INFO - Original: Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.
2026-05-10 12:33:43,566 - missing_citations - INFO - Resolved: ResolvedReference(raw_reference='Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.', resolved_paper_id='W2880875857', openalex_id='W2880875857', title='Contextual String Embeddings for Sequence Labeling', doi=None, method='openalex', confidence=1.0, unresolved_reason=None)
---
2026-05-10 12:33:43,566 - missing_citations - INFO - Original: Rami Al-R

In [5]:
if resolver.stats.get("openalex_external", -1) not in [-1, 0]:
    from sentence_transformers import SentenceTransformer
    from fastembed import SparseTextEmbedding
    from database.qdrant import create_qdrant_client
    from utils.config import config

    logger.info("Initializing models and Qdrant client...")

    # Initialize Qdrant client
    qdrant_client = create_qdrant_client(config.QDRANT_URL)
    logger.info(f"Connected to Qdrant at {config.QDRANT_URL}")

    # Load dense model
    logger.info(f"Loading dense model: {config.DENSE_MODEL}...")
    dense_model = SentenceTransformer(config.DENSE_MODEL)
    logger.info("Dense model loaded")

    # Load sparse model
    logger.info(f"Loading sparse model: {config.SPARSE_MODEL}...")
    sparse_model = SparseTextEmbedding(model_name=config.SPARSE_MODEL)
    logger.info("Sparse model loaded")

In [6]:
from ingest_missing import ingest_openalex_papers
from indexer import EmbeddingIndex

# 1. Collect the IDs of all papers that were found externally
missing_ids = [
    ref.openalex_id 
    for ref in resolved_references 
    if ref.method == "openalex_external" and ref.openalex_id
]

if missing_ids:
    # 2. You will need to pass your initialized EmbeddingIndex. 
    # (Assuming you already have your qdrant_client, dense_model, etc. initialized)
    embedding_idx = EmbeddingIndex(
        qdrant_client=qdrant_client,
        dense_model=dense_model,
        sparse_model=sparse_model
    )
    
    # 3. Fetch, insert to Postgres, embed, and insert to Qdrant!
    inserted_count = ingest_openalex_papers(missing_ids, embedding_idx)
    print(f"Successfully ingested {inserted_count} missing papers into the local corpus.")


In [ ]:
"""
STAGE 4A - Citation Worthiness Classification with GEMINI CLASSIFIER
"""

logger.info("Classifying sentences for citation worthiness using Gemini Classifier...")
citation_classifier = CitationClassifier(model=config.CLASSIFIER_BACKUP[2], batch_size=31)

classified_sentences = citation_classifier.classify_sentences(sentences[:60], parsed_paper.title, parsed_paper.abstract)

logger.info("Classification results:")
for i, sentence in enumerate(classified_sentences):
    logger.info(f"Classification {i}: {sentence.__dict__}")
    print("---")

In [8]:
"""
STAGE 4B - Urgency Scorer
"""


# 1. Connect to Qdrant Database
client = create_qdrant_client(config.QDRANT_URL)

# 2. Load the Embedding Models (this might take a moment if they aren't downloaded)
dense_model = SentenceTransformer(config.DENSE_MODEL)
sparse_model = SparseTextEmbedding(model_name=config.SPARSE_MODEL)

# 3. Initialize the Hybrid Retriever
retriever = HybridRetriever(
    qdrant_client=client,
    dense_model=dense_model,
    sparse_model=sparse_model,
    collection=config.QDRANT_COLLECTION_NAME,
    prefetch_limit=40, # Number of candidates fetched before RRF fusion
)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2440.80it/s]


In [9]:
scorer = UrgencyScorer(retriever=retriever)
scored_sentences, features = scorer.score_sentences(classified_sentences[:60], 2020)

In [10]:
for i, s in enumerate(scored_sentences[:60]):
    print(s)

print(features)

SentenceRecord(text='We introduce a new language representation model called BERT...', section='Abstract', pos=0.00, has_cite=False, citation_intent=METHOD, citation_state=NOT_CITATION_WORTHY, worthiness_score=0.0, urgency_score=None)
SentenceRecord(text='Unlike recent language representation models (Peters et al.,...', section='Abstract', pos=0.25, has_cite=True, citation_intent=METHOD, citation_state=HAS_CITATION, worthiness_score=0.9, urgency_score=None)
SentenceRecord(text='As a result, the pre-trained BERT model can be fine-tuned wi...', section='Abstract', pos=0.50, has_cite=False, citation_intent=RESULT, citation_state=NOT_CITATION_WORTHY, worthiness_score=0.0, urgency_score=None)
SentenceRecord(text='BERT is conceptually simple and empirically powerful.', section='Abstract', pos=0.75, has_cite=False, citation_intent=OTHER, citation_state=NOT_CITATION_WORTHY, worthiness_score=0.0, urgency_score=None)
SentenceRecord(text='It obtains new state-of-the-art results on eleven natural 

In [ ]:
# STAGE 5 - Claim Decomposer

